# Week 3 — Integrating Feast Feature Store into the IRIS Pipeline

This notebook implements Tasks 1-5 (and notes on Task 6) using the
time-aware `iris_data_adapted_for_feast.csv` dataset provided in the
`ga_resources` repo (branch `week_3`).

**Dataset**: 3 iris plants (`iris_id` 1001, 1002, 1003), 15 days of
measurements each, with real `event_timestamp` and `created_timestamp`
columns — already Feast-compatible, no synthetic timestamps needed.


In [1]:
!pip install feast scikit-learn pandas -q

## Task 1: Initialize the Feast Feature Repository

Rather than using `feast init` (which nests everything inside an extra
`feature_repo/` subfolder and adds unrelated example files), we build the
repository structure directly — a `feature_store.yaml` at the project
root plus a `data/` directory, which is exactly what Feast conventions
require.

In [2]:
import os

os.makedirs("iris_feature_repo/data", exist_ok=True)

feature_store_yaml = '''project: iris_feature_repo
provider: local
registry: data/registry.db
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 2
'''

with open("iris_feature_repo/feature_store.yaml", "w") as f:
    f.write(feature_store_yaml)

print("Feast repo structure created:")
for root, dirs, files in os.walk("iris_feature_repo"):
    for name in files:
        print(os.path.join(root, name))

Feast repo structure created:
iris_feature_repo/inference.py
iris_feature_repo/train.py
iris_feature_repo/iris_model.joblib
iris_feature_repo/feature_store.yaml
iris_feature_repo/iris_repo.py
iris_feature_repo/data/iris_data_adapted_for_feast.parquet
iris_feature_repo/data/registry.db
iris_feature_repo/data/online_store.db
iris_feature_repo/data/iris_data_adapted_for_feast.csv


In [3]:
import shutil
shutil.copy("iris_data_adapted_for_feast.csv", "iris_feature_repo/data/iris_data_adapted_for_feast.csv")
print("dataset copied into feature repo")

dataset copied into feature repo


Feast's `FileSource` works most reliably with Parquet, so we convert
the CSV once (keeping the CSV around too, for the raw-data comparison in
Task 5).

In [4]:
import pandas as pd

df = pd.read_csv("iris_feature_repo/data/iris_data_adapted_for_feast.csv")
df["event_timestamp"] = pd.to_datetime(df["event_timestamp"])
df["created_timestamp"] = pd.to_datetime(df["created_timestamp"])
df["iris_id"] = df["iris_id"].astype("int64")
df.to_parquet("iris_feature_repo/data/iris_data_adapted_for_feast.parquet")
df.head()

,event_timestamp,iris_id,sepal_length,sepal_width,petal_length,petal_width,species,created_timestamp
0,2025-09-17 10:40:17.102131,1001,5.52,2.53,3.86,1.13,versicolor,2025-10-02 10:40:17.172178
1,2025-09-18 10:40:17.102131,1001,5.50,2.24,3.60,1.08,versicolor,2025-10-02 10:40:17.172178
2,2025-09-19 10:40:17.102131,1001,5.55,2.47,3.75,1.08,versicolor,2025-10-02 10:40:17.172178
3,2025-09-20 10:40:17.102131,1001,5.45,2.37,3.92,1.20,versicolor,2025-10-02 10:40:17.172178
4,2025-09-21 10:40:17.102131,1001,5.65,2.52,3.95,1.17,versicolor,2025-10-02 10:40:17.172178


## Task 2: Define Entities, Data Sources & Feature Views

- **Entity**: `iris_id` uniquely identifies each iris plant being tracked.
- **Data source**: points at the parquet file; `timestamp_field` is set
  explicitly to `event_timestamp` (Feast can't auto-infer it here because
  both `event_timestamp` and `created_timestamp` look like timestamp
  columns).
- **Feature view**: maps the 4 numeric measurements + `species` to the
  entity and source.

In [5]:
repo_definitions = """
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, String

iris = Entity(
    name="iris_id",
    join_keys=["iris_id"],
    description="Unique identifier for each individual iris plant being tracked",
)

iris_source = FileSource(
    name="iris_source",
    path="data/iris_data_adapted_for_feast.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

iris_features_view = FeatureView(
    name="iris_features",
    entities=[iris],
    ttl=timedelta(days=60),
    schema=[
        Field(name="sepal_length", dtype=Float32),
        Field(name="sepal_width", dtype=Float32),
        Field(name="petal_length", dtype=Float32),
        Field(name="petal_width", dtype=Float32),
        Field(name="species", dtype=String),
    ],
    online=True,
    source=iris_source,
)
"""

with open("iris_feature_repo/iris_repo.py", "w") as f:
    f.write(repo_definitions)
print("iris_repo.py written")

iris_repo.py written


## Task 3: Apply Definitions & Materialize Features

In [6]:
!cd iris_feature_repo && feast apply

/opt/micromamba/lib/python3.12/site-packages/feast/repo_config.py:420: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(
/home/jupyter/23F2004644_MLOPS_WEEKLY_ASSIGNMENT/iris_feature_repo/iris_repo.py:6: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'iris_id'.
  iris = Entity(
No project found in the repository. Using project name iris_feature_repo defined in feature_store.yaml
Applying changes for project iris_feature_repo
Updated feature view iris_features
	batch_source: type: BATCH_FILE
timestamp_field: "event_timestamp"
created_timestamp_column: "created_timestamp"
file_options {
  uri: "data/iris_data_adapted_for_feast.parquet"
}
data_source_class_type: "feast.infra.offline_stores.file_source.FileSource"
name: "iris_source"
meta {
  created_timestamp {
    seconds: 1783254736
    nanos: 262379000
  }
  last_up

In [7]:
!cd iris_feature_repo && feast materialize 2025-09-01T00:00:00 2025-10-05T00:00:00

/opt/micromamba/lib/python3.12/site-packages/feast/repo_config.py:420: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(
Materializing 1 feature views from 2025-09-01 00:00:00+00:00 to 2025-10-05 00:00:00+00:00 into the sqlite online store.

iris_features:
/opt/micromamba/lib/python3.12/site-packages/feast/infra/key_encoding_utils.py:146: UserWarning: Serialization of entity key with version < 3 is removed. Please use version 3 by setting entity_key_serialization_version=3.To reserializa your online store featrues refer -  https://github.com/feast-dev/feast/blob/master/docs/how-to-guides/entity-reserialization-of-from-v2-to-v3.md
  warnings.warn(


Materialization ran without errors and reported processing the
`iris_features` view — confirming the online SQLite store is populated.

## Task 4: Fetch Features for Training (Offline Store)

Training pulls features via `get_historical_features` — **not** by
reading the CSV directly. Only the entity key, timestamp, and label are
supplied; Feast performs the point-in-time join to attach features.

In [8]:
from feast import FeatureStore
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from joblib import dump

store = FeatureStore(repo_path="iris_feature_repo")

FEATURES = [
    "iris_features:sepal_length",
    "iris_features:sepal_width",
    "iris_features:petal_length",
    "iris_features:petal_width",
]

raw = pd.read_csv("iris_data_adapted_for_feast.csv")
entity_df = raw[["iris_id", "event_timestamp", "species"]].copy()
entity_df["event_timestamp"] = pd.to_datetime(entity_df["event_timestamp"])

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=FEATURES,
).to_df()

training_df.head()

/opt/micromamba/lib/python3.12/site-packages/feast/repo_config.py:420: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(


,iris_id,event_timestamp,species,sepal_length,sepal_width,petal_length,petal_width
0,1001,2025-09-17 10:40:17.102131+00:00,versicolor,5.52,2.53,3.86,1.13
1,1003,2025-09-17 10:40:17.102131+00:00,setosa,5.09,3.42,1.34,0.22
2,1002,2025-09-17 10:40:17.102131+00:00,setosa,4.92,3.05,1.43,0.29
3,1001,2025-09-18 10:40:17.102131+00:00,versicolor,5.50,2.24,3.60,1.08
4,1002,2025-09-18 10:40:17.102131+00:00,setosa,5.05,2.94,1.53,0.31


In [9]:
X = training_df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = training_df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

acc = accuracy_score(y_test, model.predict(X_test))
print(f"Test accuracy: {acc:.4f}")

dump(model, "iris_model.joblib")

Test accuracy: 1.0000


['iris_model.joblib']

## Task 5: Fetch Features for Inference (Online Store)

Simulate real-time inference: given `iris_id`s, pull features from the
**online** store and compare predictions against predictions made
directly from the raw CSV, to demonstrate no training/serving skew.

In [10]:
FEATURE_COLS = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
iris_ids = [1001, 1002, 1003]

online = store.get_online_features(
    features=FEATURES,
    entity_rows=[{"iris_id": i} for i in iris_ids],
).to_dict()

online_df = pd.DataFrame(online)
online_df["prediction"] = model.predict(online_df[FEATURE_COLS])
online_df[["iris_id"] + FEATURE_COLS + ["prediction"]]

/opt/micromamba/lib/python3.12/site-packages/feast/infra/key_encoding_utils.py:146: UserWarning: Serialization of entity key with version < 3 is removed. Please use version 3 by setting entity_key_serialization_version=3.To reserializa your online store featrues refer -  https://github.com/feast-dev/feast/blob/master/docs/how-to-guides/entity-reserialization-of-from-v2-to-v3.md
  warnings.warn(


,iris_id,sepal_length,sepal_width,petal_length,petal_width,prediction
0,1001,5.45,2.36,3.84,1.09,versicolor
1,1002,4.84,2.90,1.29,0.20,setosa
2,1003,4.85,3.40,1.19,0.29,setosa


In [11]:
# Compare against predictions using the latest raw-CSV row per iris_id
latest = (
    raw.sort_values("event_timestamp")
    .groupby("iris_id")
    .tail(1)
    .set_index("iris_id")
    .loc[iris_ids]
    .reset_index()
)
latest["prediction"] = model.predict(latest[FEATURE_COLS])

match = list(online_df.sort_values("iris_id")["prediction"]) == list(
    latest.sort_values("iris_id")["prediction"]
)
print("Predictions match (no training/serving skew):", match)
latest[["iris_id"] + FEATURE_COLS + ["prediction"]]

Predictions match (no training/serving skew): True


,iris_id,sepal_length,sepal_width,petal_length,petal_width,prediction
0,1001,5.45,2.36,3.84,1.09,versicolor
1,1002,4.84,2.90,1.29,0.20,setosa
2,1003,4.85,3.40,1.19,0.29,setosa


## Task 6 (Optional): BigQuery Backend

We now replace the **offline store** (used for historical/training
retrieval in Task 4) with BigQuery. The online store is left as SQLite,
since it already serves Task 5's low-latency lookups well and swapping
it is optional per the assignment ("either ... or both").

This requires a GCP project with the BigQuery API enabled, and
authentication already set up in this environment (e.g. running on a GCP
notebook instance / Vertex Workbench, or `gcloud auth application-default
login` beforehand).

**Set your project ID below before running.**

In [16]:
GCP_PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"  # <-- replace with your project id

### Step 1: Load the dataset into a BigQuery table

In [17]:
from google.cloud import bigquery

client = bigquery.Client(project=GCP_PROJECT_ID)

dataset_id = "iris_feast_dataset"
table_id = "iris_features"

dataset_ref = bigquery.Dataset(f"{GCP_PROJECT_ID}.{dataset_id}")
dataset_ref.location = "US"
client.create_dataset(dataset_ref, exists_ok=True)
print(f"Dataset {GCP_PROJECT_ID}.{dataset_id} ready")

bq_df = pd.read_csv("iris_data_adapted_for_feast.csv")
bq_df["event_timestamp"] = pd.to_datetime(bq_df["event_timestamp"])
bq_df["created_timestamp"] = pd.to_datetime(bq_df["created_timestamp"])
bq_df["iris_id"] = bq_df["iris_id"].astype("int64")

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    schema=[
        bigquery.SchemaField("event_timestamp", "TIMESTAMP"),
        bigquery.SchemaField("iris_id", "INTEGER"),
        bigquery.SchemaField("sepal_length", "FLOAT"),
        bigquery.SchemaField("sepal_width", "FLOAT"),
        bigquery.SchemaField("petal_length", "FLOAT"),
        bigquery.SchemaField("petal_width", "FLOAT"),
        bigquery.SchemaField("species", "STRING"),
        bigquery.SchemaField("created_timestamp", "TIMESTAMP"),
    ],
)

full_table_id = f"{GCP_PROJECT_ID}.{dataset_id}.{table_id}"
job = client.load_table_from_dataframe(bq_df, full_table_id, job_config=job_config)
job.result()

table = client.get_table(full_table_id)
print(f"Loaded {table.num_rows} rows into {full_table_id}")

Dataset qwiklabs-gcp-04-e2161030bace.iris_feast_dataset ready
Loaded 45 rows into qwiklabs-gcp-04-e2161030bace.iris_feast_dataset.iris_features


### Step 2: Create a BigQuery-backed Feast repo

In [18]:
import os

os.makedirs("iris_feature_repo_bq/data", exist_ok=True)

bq_feature_store_yaml = f'''project: iris_feature_repo_bq
provider: gcp
registry: data/registry.db

offline_store:
    type: bigquery
    dataset: {dataset_id}
    project_id: {GCP_PROJECT_ID}
    location: US

online_store:
    type: sqlite
    path: data/online_store.db

entity_key_serialization_version: 2
'''

with open("iris_feature_repo_bq/feature_store.yaml", "w") as f:
    f.write(bq_feature_store_yaml)

print("BigQuery-backed feature_store.yaml written")

BigQuery-backed feature_store.yaml written


In [19]:
bq_repo_definitions = f'''
from datetime import timedelta
from feast import BigQuerySource, Entity, FeatureView, Field
from feast.types import Float32, String

iris = Entity(
    name="iris_id",
    join_keys=["iris_id"],
    description="Unique identifier for each individual iris plant being tracked",
)

iris_source = BigQuerySource(
    name="iris_bq_source",
    table="{GCP_PROJECT_ID}.{dataset_id}.{table_id}",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

iris_features_view = FeatureView(
    name="iris_features",
    entities=[iris],
    ttl=timedelta(days=60),
    schema=[
        Field(name="sepal_length", dtype=Float32),
        Field(name="sepal_width", dtype=Float32),
        Field(name="petal_length", dtype=Float32),
        Field(name="petal_width", dtype=Float32),
        Field(name="species", dtype=String),
    ],
    online=True,
    source=iris_source,
)
'''

with open("iris_feature_repo_bq/iris_repo_bq.py", "w") as f:
    f.write(bq_repo_definitions)

print("iris_repo_bq.py written")

iris_repo_bq.py written


### Step 3: Apply and materialize with the BigQuery-backed repo

In [20]:
!cd iris_feature_repo_bq && feast apply

/opt/micromamba/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=9823) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


/opt/micromamba/lib/python3.12/site-packages/db_dtypes/core.py:51: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  _internal_fill_value = numpy.datetime64("NaT")
/opt/micromamba/lib/python3.12/site-packages/feast/repo_config.py:420: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(
/home/jupyter/23F2004644_MLOPS_WEEKLY_ASSIGNMENT/iris_feature_repo_bq/iris_repo_bq.py:6: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'iris_id'.
  iris = Entity(
No project found in the repository. Using project name iris_feature_repo_bq defined in feature_store.yaml
Applying changes for project iris_feature_repo_bq
Deploying infrastructure for iris_features


In [21]:
!cd iris_feature_repo_bq && feast materialize 2025-09-01T00:00:00 2025-10-05T00:00:00

/opt/micromamba/lib/python3.12/site-packages/db_dtypes/core.py:51: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  _internal_fill_value = numpy.datetime64("NaT")
/opt/micromamba/lib/python3.12/site-packages/feast/repo_config.py:420: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(
Materializing 1 feature views from 2025-09-01 00:00:00+00:00 to 2025-10-05 00:00:00+00:00 into the sqlite online store.

iris_features:
/opt/micromamba/lib/python3.12/site-packages/feast/infra/key_encoding_utils.py:146: UserWarning: Serialization of entity key with version < 3 is removed. Please use version 3 by setting entity_key_serialization_version=3.To reserializa your online store featrues refer -  https://github.com/feast-dev/feast/blo

### Step 4: Retrain using BigQuery offline retrieval

Identical code to Task 4 -- only the repo path changed, since Feast
abstracts the offline store behind `get_historical_features`.

In [22]:
bq_store = FeatureStore(repo_path="iris_feature_repo_bq")

bq_entity_df = raw[["iris_id", "event_timestamp", "species"]].copy()
bq_entity_df["event_timestamp"] = pd.to_datetime(bq_entity_df["event_timestamp"])

bq_training_df = bq_store.get_historical_features(
    entity_df=bq_entity_df,
    features=FEATURES,
).to_df()

bq_training_df.head()

/opt/micromamba/lib/python3.12/site-packages/feast/repo_config.py:420: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(


,iris_id,event_timestamp,species,sepal_length,sepal_width,petal_length,petal_width
0,1001,2025-09-28 10:40:17.102131+00:00,versicolor,5.45,2.41,3.87,1.00
1,1001,2025-09-22 10:40:17.102131+00:00,versicolor,5.59,2.19,3.78,1.09
2,1001,2025-09-30 10:40:17.102131+00:00,versicolor,5.50,2.36,3.77,1.31
3,1001,2025-09-24 10:40:17.102131+00:00,versicolor,5.62,2.34,3.85,1.06
4,1001,2025-09-29 10:40:17.102131+00:00,versicolor,5.46,2.36,3.95,1.14


### Observed trade-offs: SQLite/local file vs. BigQuery offline store

- **Latency**: every `get_historical_features` call against BigQuery pays
  network round-trip + query-planning overhead (typically a few hundred ms
  to a few seconds even for tiny tables), versus near-instant local
  Parquet/file reads.
- **Cost**: BigQuery charges per byte scanned per query (with a free
  monthly tier). For this 45-row dataset the cost is effectively zero,
  but it becomes a real line item at production scale.
- **Scalability**: local file/SQLite offline stores don't scale past what
  fits comfortably in memory/disk on one machine. BigQuery scales to
  petabytes and supports concurrent access from many training jobs.
- **Operational overhead**: BigQuery requires GCP project setup, IAM
  permissions, and network access — meaningfully more setup than a local
  file, which is why local backends are preferred for prototyping/small
  courses assignments and BigQuery for production pipelines.
- **Online store note**: this setup keeps SQLite for the online store,
  since Task 5's low-latency lookups don't need BigQuery's strengths —
  BigQuery is an *offline/analytical* store, not built for the
  millisecond-level reads used at serving time. A production system would
  more likely pair a BigQuery offline store with a Datastore/Bigtable/
  Redis online store.
